# Lab 06 — Preparing data for an annotation pilot

Prepare a small text-classification pilot for Label Studio: **DataFrame → select examples → assign workers → export CSV**. The [analysis notebook](02_analyze_pilot.ipynb) reads the files separately.

## 1. Load and prepare the data

Load the dataset and decode its labels. IDs are assigned from the original training-row positions before filtering or sampling. Adapt these pandas steps to your own dataset.

### 1.1 Load the dataset
We use our BANKING77 dataset, with the customer support messages. We have it locally. 
We only work with the training split.

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("../../data/PolyAI_banking77")
train = dataset["train"].to_pandas()

train.head()

### 1.2 Prepare the labels
The dataset stores labels as numbers. We recover their names and map them to the nine intents used in our course.

In [ ]:
BANKING_INTENTS = {
    "passcode_forgotten": "account_access_passcode",
    "pending_transfer": "bank_transfer_pending",
    "transfer_not_received_by_recipient": "bank_transfer_not_received",
    "pending_card_payment": "card_payment_pending",
    "declined_card_payment": "card_payment_declined",
    "transaction_charged_twice": "card_payment_duplicated",
    "pending_cash_withdrawal": "cash_withdrawal_pending",
    "cash_withdrawal_not_recognised": "cash_withdrawal_unrecognized",
    "card_acceptance": "card_acceptance",
}

label_names = dataset["train"].features["label"].names

train["original_label"] = train["label"].map(dict(enumerate(label_names)))
train["reference_label"] = train["original_label"].map(BANKING_INTENTS)

# Keep only the intents used in our course
pool = train[train["reference_label"].notna()].copy()

pool[["text", "original_label", "reference_label"]].head()

### 1.3 Add an identifier
Each example needs a unique ID so we can match the annotation results with the original dataset.

In [ ]:
pool["example_id"] = "train:" + pool.index.astype(str)

print(f"Examples available: {len(pool)}")
print(f"Number of intents: {pool['reference_label'].nunique()}")

pool[["example_id", "text", "reference_label"]].head()

## 2. Select examples for the pilot

Select a small set of examples to annotate. You can use random sampling, choose representative examples, or deliberately include difficult cases.

For simplicity, we select 18 examples randomly from our dataset.

In [ ]:
SAMPLE_SIZE = 18
RANDOM_SEED = 42

# Select examples for the pilot
selected = pool.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED)

selected[["example_id", "text", "reference_label"]]

## 3. Assign examples to annotators

Label Studio Community Edition has limited support for assigning specific tasks to annotators. We can work around this by preparing assignments ourselves and using filtered tabs in Label Studio.

The `prepare_for_assignment()` function distributes our selected examples among workers. We specify:

- **Number of workers:** How many people will annotate.
- **Ratings per example:** How many different people should annotate each example.

Each example is assigned to the requested number of workers, and the workload is distributed as evenly as possible.

For example, with 18 examples, 3 workers, and 2 ratings per example, each worker receives 12 examples.

In [ ]:
from utils.assignment import prepare_for_assignment

NUM_WORKERS = 3
NUM_RATINGS_PER_EXAMPLE = 2

selected, tasks, assignments = prepare_for_assignment(
    selected,
    id_col="example_id",
    text_col="text",
    num_workers=NUM_WORKERS,
    num_ratings_per_example=NUM_RATINGS_PER_EXAMPLE,
    seed=42,
)

tasks.head()

## 4. Export and use the files

We export three CSV files:

- **`pilot_tasks.csv`:** Import this into Label Studio. Contains the text, example IDs, and worker assignments.
- **`pilot_reference.csv`:** Keep the original labels for comparison after the pilot. Do not share these labels with annotators.
- **`pilot_assignments.csv`:** Records which worker should annotate each example.

### Assign tasks in Label Studio

1. Import `pilot_tasks.csv` into your project.
2. In the Data Manager, create a new tab for each worker.
3. Filter the `assigned_workers` field using **contains** `|worker_01|` for the first worker, `|worker_02|` for the second, etc.
4. Ask each worker to open their tab and select **Label Tasks As Displayed**.

The `|` characters allow us to match each worker ID precisely.

**Important:** Make sure the project allows at least as many annotations per task as specified by `NUM_RATINGS_PER_EXAMPLE`. These assignments help organize the work but do not prevent annotators from accessing other tasks.

In [ ]:
from pathlib import Path

# We put the files in the 'export' folder
EXPORT_DIR = Path("export")
EXPORT_DIR.mkdir(exist_ok=True)

# Export tasks for Label Studio
tasks.to_csv(EXPORT_DIR / "pilot_tasks.csv", index=False)

# Save the original labels separately for later comparison
selected[["example_id", "text", "reference_label"]].to_csv(
    EXPORT_DIR / "pilot_reference.csv", index=False
)

# Save the worker assignments
assignments.to_csv(EXPORT_DIR / "pilot_assignments.csv", index=False)

## 5. After the pilot

Export the annotations from Label Studio in **JSON format**. We will use the next notebook to analyze the results and compare them with our original reference labels.

For more information, see the [Label Studio Data Manager guide](https://labelstud.io/guide/manage_data.html).